# Neural Machine Translation using LSTMs with Attention Mechanism

Welcome! In this project, I will will build an English-to-Portuguese neural machine translation (NMT) model using Long Short-Term Memory (LSTM) networks with attention. Implementing this using just a Recurrent Neural Network (RNN) with LSTMs can work for short to medium length sentences but can result in `vanishing gradients` for very long sequences. To help with this, you will be adding an `attention mechanism` to allow the decoder to access all relevant parts of the input sentence regardless of its length. 

Following are the key points of this project:

- Implement an encoder-decoder system with attention
- Build the NMT model from scratch using Tensorflow
- Generate translations using greedy and Minimum Bayes Risk (MBR) decoding

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Setting this env variable prevents TF warnings from showing up

import numpy as np
import tensorflow as tf
from collections import Counter
from utils import (sentences, train_data, val_data, english_vectorizer, portuguese_vectorizer, 
                   masked_loss, masked_acc, tokens_to_text)

In [2]:
import w1_unittest

<a name="1"></a>
## 1. Data Preparation

The text pre-processing bits have been taken care of in the `utils.py` file. You can visit that file in this repository to look at how pre-processing is happening in terms of programming logic.
The steps performed can be summarized as:

- Reading the raw data from the text files
- Cleaning the data (using lowercase, adding space around punctuation, trimming whitespaces, etc)
- Splitting it into training and validation sets
- Adding the start-of-sentence and end-of-sentence tokens to every sentence
- Tokenizing the sentences
- Creating a Tensorflow dataset out of the tokenized sentences

I will inspect the raw sentences for visualization purposes:

In [3]:
portuguese_sentences, english_sentences = sentences

print(f"English (to translate) sentence:\n\n{english_sentences[-5]}\n")
print(f"Portuguese (translation) sentence:\n\n{portuguese_sentences[-5]}")

English (to translate) sentence:

No matter how much you try to convince people that chocolate is vanilla, it'll still be chocolate, even though you may manage to convince yourself and a few others that it's vanilla.

Portuguese (translation) sentence:

Não importa o quanto você tenta convencer os outros de que chocolate é baunilha, ele ainda será chocolate, mesmo que você possa convencer a si mesmo e poucos outros de que é baunilha.


Deleting them to save memory for efficiency since they are not used in this project (I only printed them to help users visualize)

In [4]:
del portuguese_sentences
del english_sentences
del sentences

The `english_vectorizer` and `portuguese_vectorizer` from `utils.py` were created using [tf.keras.layers.TextVectorization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization) and they provide interesting features such as ways to visualize the vocabulary and convert text into tokenized ids and vice versa. 

Lets inspect the first ten words of the vocabularies for both languages:

In [5]:
print(f"First 10 words of the english vocabulary:\n\n{english_vectorizer.get_vocabulary()[:10]}\n")
print(f"First 10 words of the portuguese vocabulary:\n\n{portuguese_vectorizer.get_vocabulary()[:10]}")

First 10 words of the english vocabulary:

['', '[UNK]', '[SOS]', '[EOS]', '.', 'tom', 'i', 'to', 'you', 'the']

First 10 words of the portuguese vocabulary:

['', '[UNK]', '[SOS]', '[EOS]', '.', 'tom', 'que', 'o', 'nao', 'eu']


First 4 words are reserved for special words. In order, these are:

- the empty string
- a special token to represent an unknown word
- a special token to represent the start of a sentence
- a special token to represent the end of a sentence

We can see how many words are in a vocabulary by using the `vocabulary_size` method:

In [6]:
# Size of the vocabulary
vocab_size_por = portuguese_vectorizer.vocabulary_size()
vocab_size_eng = english_vectorizer.vocabulary_size()

print(f"Portuguese vocabulary is made up of {vocab_size_por} words")
print(f"English vocabulary is made up of {vocab_size_eng} words")

Portuguese vocabulary is made up of 12000 words
English vocabulary is made up of 12000 words


[tf.keras.layers.StringLookup](https://www.tensorflow.org/api_docs/python/tf/keras/layers/StringLookup) objects help map from words to ids and vice versa. I will do this for the portuguese vocabulary since this will be useful later on when you decode the predictions from your model

In [7]:
# words to ids
word_to_id = tf.keras.layers.StringLookup(
    vocabulary=portuguese_vectorizer.get_vocabulary(), 
    mask_token="", 
    oov_token="[UNK]"
)

# ids to words
id_to_word = tf.keras.layers.StringLookup(
    vocabulary=portuguese_vectorizer.get_vocabulary(),
    mask_token="",
    oov_token="[UNK]",
    invert=True,
)

Trying out for the special tokens and a random word:

In [8]:
unk_id = word_to_id("[UNK]")
sos_id = word_to_id("[SOS]")
eos_id = word_to_id("[EOS]")
baunilha_id = word_to_id("baunilha")

print(f"The id for the [UNK] token is {unk_id}")
print(f"The id for the [SOS] token is {sos_id}")
print(f"The id for the [EOS] token is {eos_id}")
print(f"The id for baunilha (vanilla) is {baunilha_id}")

The id for the [UNK] token is 1
The id for the [SOS] token is 2
The id for the [EOS] token is 3
The id for baunilha (vanilla) is 7079


`ONLY FOR VISUALIZATION PURPOSES`

Lets take a look at how the data that is going to be fed to the neural network looks like. Both `train_data` and `val_data` are of type `tf.data.Dataset` and are arranged in batches of 64 examples (preparation is happening inside the `utils.py` file). 

To get the first batch out of a tf dataset we use the `take` method. To get the first example out of the batch we slice the tensor and use the `numpy` method for nicer printing

In [9]:
for (to_translate, sr_translation), translation in train_data.take(1):
    print(f"Tokenized english sentence:\n{to_translate[0, :].numpy()}\n\n")
    print(f"Tokenized portuguese sentence (shifted to the right):\n{sr_translation[0, :].numpy()}\n\n")
    print(f"Tokenized portuguese sentence:\n{translation[0, :].numpy()}\n\n")

Tokenized english sentence:
[   2  210    9  146  123   38    9 1672    4    3    0    0    0    0]


Tokenized portuguese sentence (shifted to the right):
[   2 1085    7  128   11  389   37 2038    4    0    0    0    0    0
    0]


Tokenized portuguese sentence:
[1085    7  128   11  389   37 2038    4    3    0    0    0    0    0
    0]




Some important details to notice here:

- Padding has already been applied to the tensors with value `0`
- Each example consists of 3 different tensors:
    - The sentence to translate (input encoder)
    - The shifted-to-the-right translation (input decoder)
    - The translation (output decoder)
    
The first two can be considered as the features, while the third one as the target. By doing this your model can perform Teacher Forcing as you saw in the lectures.

<a name="2"></a>
## 2. NMT model with attention (OVERVIEW)

The model I have built in this notebook uses an encoder-decoder architecture. This Recurrent Neural Network (RNN) takes in a tokenized version of a sentence in its encoder, then passes it on to the decoder for translation. Just using a a regular sequence-to-sequence model with LSTMs will work effectively for short to medium sentences but will start to degrade for longer ones. The image below shows how this will be an issue for very long sentences (e.g. 100 tokens or more) because the context of the first parts of the input will have very little effect on the final vector passed to the decoder.

<img src='images/plain_rnn.png'>

Adding an attention layer to this model avoids this problem by giving the decoder access to all parts of the input sentence. To illustrate, let's just use a 4-word input sentence as shown below. Remember that a hidden state is produced at each timestep of the encoder (represented by the orange rectangles). These are all passed to the attention layer and each are given a score given the current activation (i.e. hidden state) of the decoder. For instance, let's consider the figure below where the first prediction "como" is already made. To produce the next prediction, the attention layer will first receive all the encoder hidden states (i.e. orange rectangles) as well as the decoder hidden state when producing the word "como" (i.e. first green rectangle). Given this information, it will score each of the encoder hidden states to know which one the decoder should focus on to produce the next word. As a result of training, the model might have learned that it should align to the second encoder hidden state and subsequently assigns a high probability to the word "você". If we are using greedy decoding, we will output the said word as the next symbol, then restart the process to produce the next word until we reach an end-of-sentence prediction.

<img src='images/attention_overview.png'>


There are different ways to implement attention and the one I have used is the popular Scaled Dot Product Attention:

$$Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

Through the above formula, we get a context vector at a particular timestep of the decoder. This context vector is fed to the decoder RNN to get a set of probabilities for the next predicted word. The division by square root of the keys dimensionality ($\sqrt{d_k}$) is for improving model performance. For the machine translation application, the encoder activations (i.e. encoder hidden states) will be the keys and values, while the decoder activations (i.e. decoder hidden states) will be the queries.

This complex architecture and mechanism can be implemented with just a few lines of code BUT in this project, I show how this can be implemented from scratch. 

First I define two important global variables:

- The size of the vocabulary
- The number of units in the LSTM layers (the same number will be used for all LSTM layers)

Since the vocabulary sizes for English and Portuguese are the same, we use a single constant VOCAB_SIZE throughout the notebook.

In [10]:
VOCAB_SIZE = 12000
UNITS = 256

<a name="ex1"></a>
## Encoder

I start by coding the encoder part of the neural network. For this, I will implement the `Encoder` class. In the constructor (the `__init__` method), I define all of the sublayers of the encoder and then use these sublayers during the forward pass (the `call` method).

The encoder consists of the following layers:

- [Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding). For this layer, it is important to define the appropriate `input_dim` and `output_dim` and let it know that we are using '0' as padding, which can be done by using the appropriate value for the `mask_zero` parameter.
    
+ [Bidirectional](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Bidirectional) [LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM). In TF you can implement bidirectional behaviour for RNN-like layers. For that, we need to specify the appropriate type of layer as well as its parameters. In particular, I will set the appropriate number of units and make sure that the LSTM returns the full sequence and not only the last output, which can be done by using the appropriate value for the `return_sequences` parameter.


I will define the forward pass using the syntax of TF's [functional API](https://www.tensorflow.org/guide/keras/functional_api). What this means is that we chain function calls together.

In [11]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, units):
        """Initializes an instance of this class

        Args:
            vocab_size (int): Size of the vocabulary
            units (int): Number of units in the LSTM layer
        """
        super(Encoder, self).__init__()


        self.embedding = tf.keras.layers.Embedding(  
            input_dim=vocab_size,
            output_dim=units,
            mask_zero=True
        )  

        self.rnn = tf.keras.layers.Bidirectional(  
            merge_mode="sum",  
            layer=tf.keras.layers.LSTM(
                units=units,
                return_sequences=True
            ),  
        )  


    def call(self, context):
        """Forward pass of this layer

        Args:
            context (tf.Tensor): The sentence to translate

        Returns:
            tf.Tensor: Encoded sentence to translate
        """


        # Passing the context through the embedding layer
        x = self.embedding(context)

        # Passing the output of the embedding through the RNN
        x = self.rnn(x)


        return x

In [12]:
# Doing a quick check of implementation

# Creating an instance of class
encoder = Encoder(VOCAB_SIZE, UNITS)

# Passing a batch of sentences to translate from english to portuguese
encoder_output = encoder(to_translate)

print(f'Tensor of sentences in english has shape: {to_translate.shape}\n')
print(f'Encoder output has shape: {encoder_output.shape}')

Tensor of sentences in english has shape: (64, 14)

Encoder output has shape: (64, 14, 256)


In [13]:
# Testing code!

w1_unittest.test_encoder(Encoder)

 All tests passed!


<a name="ex2"></a>
## CrossAttention Mechanism

Now, I will code the layer that will perform cross attention between the original sentences and the translations. For this, I have the `CrossAttention` class below. Notice that in the constructor (the `__init__` method) I define all of the sublayers and then use these sublayers during the forward pass (the `call` method).

The cross attention consists of the following layers:

- [MultiHeadAttention](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention). For this layer I will define the appropriate `key_dim`, which is the size of the key and query tensors. I also set the number of heads to 1 since I am not implementing multi head attention but attention between two tensors. The reason why this layer is preferred over [Attention](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Attention) is that it allows simpler code during the forward pass.
    
A couple of things to notice:
- I need a way to pass both the output of the attention alongside the shifted-to-the-right translation (since this cross attention happens in the decoder side). For this I use an [Add](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Add) layer so that the original dimension is preserved, which would not happen if I use something like a [Concatenate](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Concatenate) layer.

+ Layer normalization is also performed for better stability of the network by using a [LayerNormalization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LayerNormalization) layer.


In [14]:
class CrossAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        """Initializes an instance of this class

        Args:
            units (int): Number of units in the LSTM layer
        """
        super().__init__()


        self.mha = ( 
            tf.keras.layers.MultiHeadAttention(
                key_dim=units,
                num_heads=1
            ) 
        )  


        self.layernorm = tf.keras.layers.LayerNormalization()
        self.add = tf.keras.layers.Add()

    def call(self, context, target):
        """Forward pass of this layer

        Args:
            context (tf.Tensor): Encoded sentence to translate
            target (tf.Tensor): The embedded shifted-to-the-right translation

        Returns:
            tf.Tensor: Cross attention between context and target
        """

        # Calling the MH attention by passing in the query and value
        # For this case the query should be the translation and the value the encoded sentence to translate
        
        attn_output = self.mha(
            query=target,
            value=context
        )  

        

        x = self.add([target, attn_output])

        x = self.layernorm(x)

        return x

In [15]:
# Doing a quick check of implementation

# Creating an instance of your class
attention_layer = CrossAttention(UNITS)

# The attention layer expects the embedded sr-translation and the context
# The context (encoder_output) is already embedded so I to do this for sr_translation:
sr_translation_embed = tf.keras.layers.Embedding(VOCAB_SIZE, output_dim=UNITS, mask_zero=True)(sr_translation)

# Computing the cross attention
attention_result = attention_layer(encoder_output, sr_translation_embed)

print(f'Tensor of contexts has shape: {encoder_output.shape}')
print(f'Tensor of translations has shape: {sr_translation_embed.shape}')
print(f'Tensor of attention scores has shape: {attention_result.shape}')

Tensor of contexts has shape: (64, 14, 256)
Tensor of translations has shape: (64, 15, 256)
Tensor of attention scores has shape: (64, 15, 256)


In [16]:
# Testing code!

w1_unittest.test_cross_attention(CrossAttention)

 All tests passed!


<a name="ex3"></a>
## Decoder Implementation


Now I will implement the decoder part of the neural network by completing the `Decoder` class below. Notice that in the constructor (the `__init__` method) I need to define all of the sublayers of the decoder and then I will use these sublayers during the forward pass (the `call` method).

The decoder consists of the following layers:

- [Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding). For this layer I define the appropriate `input_dim` and `output_dim` and let it know that I am using '0' as padding, which can be done by using the appropriate value for the `mask_zero` parameter.
  
  
+ Pre-attention [LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM). Unlike in the encoder in which I used a Bidirectional LSTM, here I will use a vanilla LSTM. I make sure that I set the appropriate number of units and also make sure that the LSTM returns the full sequence and not only the last output, which can be done by using the appropriate value for the `return_sequences` parameter. It is very important that this layer returns the state since this will be needed for inference so make sure to set the `return_state` parameter accordingly. Notice that LSTM layers return state as a tuple of two tensors called `memory_state` and `carry_state`, **however these names have been changed to `hidden_state` and `cell_state` respectively for better understanding**.

- The attention layer that performs cross attention between the sentence to translate and the right-shifted translation. Here I use the `CrossAttention` layer defined in the previous exercise.

+ Post-attention [LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM). Another LSTM layer. For this one I will not return the state.

- Finally a [Dense](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense) layer. This one should have the same number of units as the size of the vocabulary since I expect it to compute the logits for every possible word in the vocabulary. I will use a `logsoftmax` activation function for this one, which I can get as [tf.nn.log_softmax](https://www.tensorflow.org/api_docs/python/tf/nn/log_softmax).



In [17]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, units):
        """Initializes an instance of this class

        Args:
            vocab_size (int): Size of the vocabulary
            units (int): Number of units in the LSTM layer
        """
        super(Decoder, self).__init__()


        # The embedding layer
        self.embedding = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=units,
            mask_zero=True
        )  

        # The RNN before attention
        self.pre_attention_rnn = tf.keras.layers.LSTM(
            units=units,
            return_sequences=True,
            return_state=True
        )  

        # The attention layer
        self.attention = CrossAttention(units)

        # The RNN after attention
        self.post_attention_rnn = tf.keras.layers.LSTM(
            units=units,
            return_sequences=True
        )  

        # The dense layer with logsoftmax activation
        self.output_layer = tf.keras.layers.Dense(
            units=vocab_size,
            activation='log_softmax'
        )  


    def call(self, context, target, state=None, return_state=False):
        """Forward pass of this layer

        Args:
            context (tf.Tensor): Encoded sentence to translate
            target (tf.Tensor): The shifted-to-the-right translation
            state (list[tf.Tensor, tf.Tensor], optional): Hidden state of the pre-attention LSTM. Defaults to None.
            return_state (bool, optional): If set to true return the hidden states of the LSTM. Defaults to False.

        Returns:
            tf.Tensor: The log_softmax probabilities of predicting a particular token
        """

        # Getting the embedding of the input
        x = self.embedding(target)

        # Passing the embedded input into the pre attention LSTM
        
        x, hidden_state, cell_state = self.pre_attention_rnn(x, initial_state=state)

        # Performing cross attention between the context and the output of the LSTM (in that order)
        x = self.attention(context, x)

        # Doing a pass through the post attention LSTM
        x = self.post_attention_rnn(x)

        # Computing the logits
        logits = self.output_layer(x)

        

        if return_state:
            return logits, [hidden_state, cell_state]

        return logits

In [18]:
# Doing a quick check of your implementation

# Creating an instance of your class
decoder = Decoder(VOCAB_SIZE, UNITS)

logits = decoder(encoder_output, sr_translation)

print(f'Tensor of contexts has shape: {encoder_output.shape}')
print(f'Tensor of right-shifted translations has shape: {sr_translation.shape}')
print(f'Tensor of logits has shape: {logits.shape}')

Tensor of contexts has shape: (64, 14, 256)
Tensor of right-shifted translations has shape: (64, 15)
Tensor of logits has shape: (64, 15, 12000)


In [19]:
# Testing code!

w1_unittest.test_decoder(Decoder, CrossAttention)

 All tests passed!


<a name="ex4"></a>
## Translator Implementation

Now I have to put together all of the layers previously coded into an actual model. For this, I have the `Translator` class below.

Remember that `train_data` will yield a tuple with the sentence to translate and the shifted-to-the-right translation, which are the "features" of the model. This means that the inputs of the network will be tuples containing context and targets.

In [20]:
class Translator(tf.keras.Model):
    def __init__(self, vocab_size, units):
        """Initializes an instance of this class

        Args:
            vocab_size (int): Size of the vocabulary
            units (int): Number of units in the LSTM layer
        """
        super().__init__()


        # Defining the encoder with the appropriate vocab_size and number of units
        self.encoder = Encoder(vocab_size, units)

        # Defining the decoder with the appropriate vocab_size and number of units
        self.decoder = Decoder(vocab_size, units)

        

    def call(self, inputs):
        """Forward pass of this layer

        Args:
            inputs (tuple(tf.Tensor, tf.Tensor)): Tuple containing the context (sentence to translate) and the target (shifted-to-the-right translation)

        Returns:
            tf.Tensor: The log_softmax probabilities of predicting a particular token
        """

        

        # In this case inputs is a tuple consisting of the context and the target, unpacking it into single variables
        context, target = inputs

        # Passing the context through the encoder
        encoded_context = self.encoder(context)

        # Computing the logits by passing the encoded context and the target to the decoder
        logits = self.decoder(encoded_context, target)

        

        return logits

In [21]:
# Doing a quick check of your implementation

# Creating an instance of your class
translator = Translator(VOCAB_SIZE, UNITS)

# Computing the logits for every word in the vocabulary
logits = translator((to_translate, sr_translation))

print(f'Tensor of sentences to translate has shape: {to_translate.shape}')
print(f'Tensor of right-shifted translations has shape: {sr_translation.shape}')
print(f'Tensor of logits has shape: {logits.shape}')

Tensor of sentences to translate has shape: (64, 14)
Tensor of right-shifted translations has shape: (64, 15)
Tensor of logits has shape: (64, 15, 12000)


In [22]:
w1_unittest.test_translator(Translator, Encoder, Decoder)

 All tests passed!


<a name="3"></a>
## Model Training 
For training, I use the `compile_and_train` function below:

In [23]:
def compile_and_train(model, epochs=20, steps_per_epoch=500):
    model.compile(optimizer="adam", loss=masked_loss, metrics=[masked_acc, masked_loss])

    history = model.fit(
        train_data.repeat(),
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_data,
        validation_steps=50,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=3)],
    )

    return model, history

In [24]:
# Train the translator (takes some minutes)

trained_translator, history = compile_and_train(translator)

Epoch 1/20
500/500 [==============================] - 52s 77ms/step - loss: 5.1347 - masked_acc: 0.2204 - masked_loss: 5.1373 - val_loss: 4.2882 - val_masked_acc: 0.3307 - val_masked_loss: 4.2891
Epoch 2/20
500/500 [==============================] - 17s 34ms/step - loss: 3.8013 - masked_acc: 0.3968 - masked_loss: 3.8022 - val_loss: 3.2104 - val_masked_acc: 0.4597 - val_masked_loss: 3.2122
Epoch 3/20
500/500 [==============================] - 16s 33ms/step - loss: 2.8496 - masked_acc: 0.5222 - masked_loss: 2.8511 - val_loss: 2.4571 - val_masked_acc: 0.5680 - val_masked_loss: 2.4578
Epoch 4/20
500/500 [==============================] - 16s 33ms/step - loss: 2.2761 - masked_acc: 0.6046 - masked_loss: 2.2769 - val_loss: 2.0158 - val_masked_acc: 0.6384 - val_masked_loss: 2.0164
Epoch 5/20
500/500 [==============================] - 16s 32ms/step - loss: 1.8927 - masked_acc: 0.6596 - masked_loss: 1.8938 - val_loss: 1.7293 - val_masked_acc: 0.6768 - val_masked_loss: 1.7299
Epoch 6/20
500/500 [

<a name="4"></a>
## Using trained model for inference


Now that the model is trained I can use it for inference. The `generate_next_token` function is meant to be used inside a for-loop, so I will feed to it the information of the previous step to generate the information of the next step. In particular I keep track of the state of the pre-attention LSTM in the decoder and if I am done with the translation. The `temperature` variable is introduced which determines how to select the next token given the predicted logits:  

In [25]:
def generate_next_token(decoder, context, next_token, done, state, temperature=0.0):
    """Generates the next token in the sequence

    Args:
        decoder (Decoder): The decoder
        context (tf.Tensor): Encoded sentence to translate
        next_token (tf.Tensor): The predicted next token
        done (bool): True if the translation is complete
        state (list[tf.Tensor, tf.Tensor]): Hidden states of the pre-attention LSTM layer
        temperature (float, optional): The temperature that controls the randomness of the predicted tokens. Defaults to 0.0.

    Returns:
        tuple(tf.Tensor, np.float, list[tf.Tensor, tf.Tensor], bool): The next token, log prob of said token, hidden state of LSTM and if translation is done
    """
    # Getting the logits and state from the decoder
    logits, state = decoder(context, next_token, state=state, return_state=True)
    
    # Trimming the intermediate dimension 
    logits = logits[:, -1, :]
        
    # If temp is 0 then next_token is the argmax of logits
    if temperature == 0.0:
        next_token = tf.argmax(logits, axis=-1)
        
    # If temp is not 0 then next_token is sampled out of logits
    else:
        logits = logits / temperature
        next_token = tf.random.categorical(logits, num_samples=1)
    
    # Trimming dimensions of size 1
    logits = tf.squeeze(logits)
    next_token = tf.squeeze(next_token)
    
    # Getting the logit of the selected next_token
    logit = logits[next_token].numpy()
    
    # Reshaping to (1,1) since this is the expected shape for text encoded as TF tensors
    next_token = tf.reshape(next_token, shape=(1,1))
    
    # If next_token is End-of-Sentence token its done
    if next_token == eos_id:
        done = True
    
    return next_token, logit, state, done

In [26]:
# PROCESS SENTENCE TO TRANSLATE AND ENCODE

# A sentence to translate
eng_sentence = "I love languages"

# Converting it to a tensor
texts = tf.convert_to_tensor(eng_sentence)[tf.newaxis]

# Vectorizing it and pass it through the encoder
context = english_vectorizer(texts).to_tensor()
context = encoder(context)

# SETTING STATE OF THE DECODER

# Next token is Start-of-Sentence since I start fresh
next_token = tf.fill((1,1), sos_id)

# Hidden and Cell states of the LSTM can be mocked using uniform samples
state = [tf.random.uniform((1, UNITS)), tf.random.uniform((1, UNITS))]

# Its not done until next token is EOS token
done = False

# Generating next token
next_token, logit, state, done = generate_next_token(decoder, context, next_token, done, state, temperature=0.5)
print(f"Next token: {next_token}\nLogit: {logit:.4f}\nDone? {done}")

Next token: [[6188]]
Logit: -18.7759
Done? False


<a name="ex5"></a>
## Translation Implementation

Now I can put everything together to translate a given sentence. For this, I have the `translate` function below. This function will take care of the following steps: 
- Process the sentence to translate and encode it

+ Set the initial state of the decoder

- Get predictions of the next token (starting with the \<SOS> token) for a maximum of iterations (in case the \<EOS> token is never returned)
    
+ Return the translated text (as a string), the logit of the last iteration (this helps measure how certain was that the sequence was translated in its totality) and the translation in token format.


In [27]:
def translate(model, text, max_length=50, temperature=0.0):
    """Translate a given sentence from English to Portuguese

    Args:
        model (tf.keras.Model): The trained translator
        text (string): The sentence to translate
        max_length (int, optional): The maximum length of the translation. Defaults to 50.
        temperature (float, optional): The temperature that controls the randomness of the predicted tokens. Defaults to 0.0.

    Returns:
        tuple(str, np.float, tf.Tensor): The translation, logit that predicted <EOS> token and the tokenized translation
    """
    # Lists to save tokens and logits
    tokens, logits = [], []

    
    # PROCESSING THE SENTENCE TO TRANSLATE
    
    # Converting the original string into a tensor
    text = tf.convert_to_tensor(text)[tf.newaxis]
    
    # Vectorizing the text using the correct vectorizer
    context = english_vectorizer(text).to_tensor()
    
    # Getting the encoded context (pass the context through the encoder)
    
    context = model.encoder(context)
    
    # INITIAL STATE OF THE DECODER
    
    # First token should be SOS token with shape (1,1)
    next_token = tf.fill((1, 1), sos_id)
    
    # Initial hidden and cell states should be tensors of zeros with shape (1, UNITS)
    state = [tf.zeros((1, UNITS)), tf.zeros((1, UNITS))]
    
    # Its done when we draw a EOS token as next token (initial state is False)
    done = False

    # Iterating for max_length iterations
    for _ in range(max_length):
        # Generating the next token
        try:
            next_token, logit, state, done = generate_next_token(
                decoder=model.decoder,
                context=context,
                next_token=next_token,
                done=done,
                state=state,
                temperature=temperature
            )
        except:
             raise Exception("Problem generating the next token")
        
        # If done then break out of the loop
        if done:
            break
        
        # Adding next_token to the list of tokens
        tokens.append(next_token)
        
        # Adding logit to the list of logits
        logits.append(logit)
    
    
    
    # Concatenating all tokens into a tensor
    tokens = tf.concat(tokens, axis=-1)
    
    # Converting the translated tokens into text
    translation = tf.squeeze(tokens_to_text(tokens, id_to_word))
    translation = translation.numpy().decode()
    
    return translation, logits[-1], tokens

Trying with 0 temperature will yield a deterministic output and is equivalent to a greedy decoding:

In [28]:
# Running this cell multiple times should return the same output since temp is 0

temp = 0.0 
original_sentence = "I love languages"

translation, logit, tokens = translate(trained_translator, original_sentence, temperature=temp)

print(f"Temperature: {temp}\n\nOriginal sentence: {original_sentence}\nTranslation: {translation}\nTranslation tokens:{tokens}\nLogit: {logit:.3f}")

Temperature: 0.0

Original sentence: I love languages
Translation: eu adoro idiomas .
Translation tokens:[[  9 564 850   4]]
Logit: -1.304


Trying with temperature = 0.7 (stochastic output):

In [29]:
# Running this cell multiple times should return different outputs since temp is not 0

temp = 0.7
original_sentence = "I love languages"

translation, logit, tokens = translate(trained_translator, original_sentence, temperature=temp)

print(f"Temperature: {temp}\n\nOriginal sentence: {original_sentence}\nTranslation: {translation}\nTranslation tokens:{tokens}\nLogit: {logit:.3f}")

Temperature: 0.7

Original sentence: I love languages
Translation: eu adoro idiomas .
Translation tokens:[[  9 564 850   4]]
Logit: -1.863


In [30]:
w1_unittest.test_translate(translate, trained_translator)

 All tests passed!


<a name="5"></a>
## Minimum Bayes-Risk (MBR) Decoding Implementation

Getting the most probable token at each step may not necessarily produce the best results (since it does not consider the order of the generated sentence). Another approach is to do Minimum Bayes Risk Decoding or MBR. The general steps to implement this are:

- Take several random samples
+ Score each sample against all other samples
- Select the one with the highest score

I will be building helper functions for these steps in the following sections.

With the ability to generate different translations by setting different temperature values I can generate a bunch of translations and then determine which one is the best candidate. I do this by using the provided `generate_samples` function. This function will return any desired number of candidate translations alongside the log-probability for each one:

In [32]:
def generate_samples(model, text, n_samples=4, temperature=0.6):
    
    samples, log_probs = [], []

    # Iterating for n_samples iterations
    for _ in range(n_samples):
        
        # Saving the logit and the translated tensor
        _, logp, sample = translate(model, text, temperature=temperature)
        
        # Saving the translated tensors
        samples.append(np.squeeze(sample.numpy()).tolist())
        
        # Saving the logits
        log_probs.append(logp)
                
    return samples, log_probs

In [33]:
samples, log_probs = generate_samples(trained_translator, 'I love languages')

for s, l in zip(samples, log_probs):
    print(f"Translated tensor: {s} has logit: {l:.3f}")

Translated tensor: [9, 522, 850, 21, 811, 4] has logit: -0.966
Translated tensor: [9, 564, 850, 4] has logit: -2.174
Translated tensor: [9, 522, 850, 4] has logit: -3.154
Translated tensor: [9, 564, 850, 4] has logit: -2.174


## Evaluation through Overlap Comparison

Now that I can generate multiple translations it is time to measure the goodness of each one by comparing each sample against the others. 

There are several metrics for this purpose, but I will be calculating scores for **unigram overlaps** and one of these metrics is the widely used yet simple [Jaccard similarity](https://en.wikipedia.org/wiki/Jaccard_index) which gets the intersection over union of two sets. The `jaccard_similarity` function returns this metric for any pair of candidate and reference translations:


In [34]:
def jaccard_similarity(candidate, reference):
        
    # Converting the lists to sets to get the unique tokens
    candidate_set = set(candidate)
    reference_set = set(reference)
    
    # Getting the set of tokens common to both candidate and reference
    common_tokens = candidate_set.intersection(reference_set)
    
    # Getting the set of all tokens found in either candidate or reference
    all_tokens = candidate_set.union(reference_set)
    
    # Computing the percentage of overlap (divide the number of common tokens by the number of all tokens)
    overlap = len(common_tokens) / len(all_tokens)
        
    return overlap

<a name="ex6"></a>
## rouge1_similarity 

Jaccard similarity is good but a more commonly used metric in machine translation is the ROUGE score. For unigrams, this is called ROUGE-1 (N = 1), I can output the scores for both precision and recall when comparing two samples. To get the final score, I will want to compute the F1-score which is given by:

$$score = 2* \frac{(precision * recall)}{(precision + recall)}$$

For the implementation of the `rouge1_similarity` function I will be using the [Counter](https://docs.python.org/3/library/collections.html#collections.Counter) class from the Python standard library:

In [36]:
def rouge1_similarity(candidate, reference):
    """Computes the ROUGE 1 score between two token lists

    Args:
        candidate (list[int]): Tokenized candidate translation
        reference (list[int]): Tokenized reference translation

    Returns:
        float: Overlap between the two token lists
    """
    

    # The Counter method created a frequency dictionary
    candidate_word_counts = Counter(candidate)
    reference_word_counts = Counter(reference)
    
    # Initializing overlap at 0
    overlap = 0
    
    # Iterating over the tokens in the candidate frequency table
    
    for token in candidate_word_counts.keys():
        
        # Getting the count of the current token in the candidate frequency table
        
        token_count_candidate = candidate_word_counts[token]
        
        # Getting the count of the current token in the reference frequency table
        
        token_count_reference = reference_word_counts.get(token, 0)
        
        # Updating the overlap by getting the minimum between the two token counts above
        overlap += min(token_count_candidate, token_count_reference)
    
    # Computing the precision = overlap / (number of tokens in candidate list) 

    precision = overlap / len(candidate) if len(candidate) > 0 else 0
    
    # Computing the recall = overlap / (number of tokens in reference list) 

    recall = overlap / len(reference) if len(reference) > 0 else 0
    
    if precision + recall != 0:
        # Computing the Rouge1 Score = F1 Score
        
        f1_score = 2 * (precision * recall) / (precision + recall)
        
        return f1_score
    
        
    return 0 # If precision + recall = 0 then return 0

In [38]:
w1_unittest.test_rouge1_similarity(rouge1_similarity)

 All tests passed!


## Computing the Overall Score


I now build a function to generate the overall score for a particular sample. For this, I need to compare each sample with all other samples. For instance, if we generated 30 sentences, we will need to compare sentence 1 to sentences 2 through 30. Then, we compare sentence 2 to sentences 1 and 3 through 30, and so forth (repeat 30 times). At each step, we get the average score of all comparisons to get the overall score for a particular sample. To illustrate, these will be the steps to generate the scores of a 4-sample list.

- Get similarity score between sample 1 and sample 2
+ Get similarity score between sample 1 and sample 3
- Get similarity score between sample 1 and sample 4
+ Get average score of the first 3 steps. This will be the overall score of sample 1
- Iterate and repeat until samples 1 to 4 have overall scores.


The results will be stored in a dictionary for easy lookups.

<a name="ex7"></a>
## average_overlap() function

This function implements the process described above

In [39]:
def average_overlap(samples, similarity_fn):
    """Computes the arithmetic mean of each candidate sentence in the samples

    Args:
        samples (list[list[int]]): Tokenized version of translated sentences
        similarity_fn (Function): Similarity function used to compute the overlap

    Returns:
        dict[int, float]: A dictionary mapping the index of each translation to its score
    """
    # Initializing dictionary
    scores = {}
    
    # Iterating through all samples (enumerate helps keep track of indexes)
    for index_candidate, candidate in enumerate(samples):    
        
        
                
        # Initially overlap is zero
        overlap = 0
        
        # Iterating through all samples (enumerate helps keep track of indexes)
        for index_sample, sample in enumerate(samples):

            # Skip if the candidate index is the same as the sample index
            if index_candidate == index_sample:
                continue
                
            # Getting the overlap between candidate and sample using the similarity function
            sample_overlap = similarity_fn(candidate, sample)
            
            # Adding the sample overlap to the total overlap
            overlap += sample_overlap

        
        # Getting the score for the candidate by computing the average
        score = overlap / (len(samples) - 1)

        # Only use 3 decimal points
        score = round(score, 3)
        
        # Saving the score in the dictionary. use index as the key.
        scores[index_candidate] = score
        
    return scores

In [42]:
w1_unittest.test_average_overlap(average_overlap)

 All tests passed!


In practice, it is also common to see the weighted mean being used to calculate the overall score instead of just the arithmetic mean. I implement this in the `weighted_avg_overlap` function below:

In [43]:
def weighted_avg_overlap(samples, log_probs, similarity_fn):
    
    # Scores dictionary
    scores = {}
    
    # Iterating over the samples
    for index_candidate, candidate in enumerate(samples):    
        
        # Initializing overlap and weighted sum
        overlap, weight_sum = 0.0, 0.0
        
        # Iterating over all samples and log probabilities
        for index_sample, (sample, logp) in enumerate(zip(samples, log_probs)):

            # Skipping if the candidate index is the same as the sample index            
            if index_candidate == index_sample:
                continue
                
            # Converting log probability to linear scale
            sample_p = float(np.exp(logp))

            # Updating the weighted sum
            weight_sum += sample_p

            # Getting the unigram overlap between candidate and sample
            sample_overlap = similarity_fn(candidate, sample)
            
            # Updating the overlap
            overlap += sample_p * sample_overlap
            
        # Computing the score for the candidate
        score = overlap / weight_sum

        # Only use 3 decimal points
        score = round(score, 3)
        
        # Saving the score in the dictionary. use index as the key.
        scores[index_candidate] = score
    
    return scores

## mbr_decode() function

I will now put everything together in the the `mbr_decode` function below. 

In [45]:
def mbr_decode(model, text, n_samples=5, temperature=0.6, similarity_fn=jaccard_similarity):
    
    # Generating samples
    samples, log_probs = generate_samples(model, text, n_samples=n_samples, temperature=temperature)
    
    # Computing the overlap scores
    scores = weighted_avg_overlap(samples, log_probs, similarity_fn)

    # Decoding samples
    decoded_translations = [tokens_to_text(s, id_to_word).numpy().decode('utf-8') for s in samples]
    
    # Finding the key with the highest score
    max_score_key = max(scores, key=lambda k: scores[k])
    
    # Getting the translation 
    translation = decoded_translations[max_score_key]
    
    return translation, decoded_translations

In [46]:
english_sentence = "I love languages"

translation, candidates = mbr_decode(trained_translator, english_sentence, n_samples=10, temperature=0.6)

print("Translation candidates:")
for c in candidates:
    print(c)

print(f"\nSelected translation: {translation}")

Translation candidates:
eu adoro idiomas .
eu amo idiomas .
eu amo idiomas fogo .
eu adoro idiomas .
eu adoro idiomas .
eu adoro idiomas .
eu amo linguas a regiao . eu eu gosto de crianca eu sinto sorte eu amo eu adoro amor eu adoro sorte eu adoro sorte eu eu ama idiomas eu adoro sorte eu eu adoro idiomas eu adoro idiomas eu amo idiomas eu adoro cuidado . eu eu queria sorte
eu amo idiomas por amor . eu eu gosto ! eu eu gosto de sorte eu amo sorte eu amo que eu eu amo luz sorte eu adoro sorte eu adoro sorte eu adoro sorte eu adoro sorte eu adoro sorte eu adoro sorte eu amo idiomas eu eu amava
eu adorei quando eu gosto de vez de ordem sabiamente eu adoro eu amo eu adoro amo adoro util eu eu adoro sorte eu adoro sorte eu amo idiomas eu amo idiomas eu eu amo sorte eu amo idiomas eu [UNK] noticias eu eu amor adoro idiomas eu adoro sorte
eu adoro idiomas .

Selected translation: eu amo idiomas .
